# Подготовка шести карт ambientCG

Техническая проверка экспорта из Parquet в RGB PNG. В этом прогоне каждый ID образует отдельную группу. Для обучающей выборки родство вариантов требуется проверить отдельно. Первая ячейка содержит снимок `texgen.dataset.prepare`.

In [1]:
"""Подготовка выборки Color по проверенному каталогу материалов ambientCG."""
import argparse
import csv
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from PIL import Image


def prepare(catalog, output, limit=400, seed=42):
    """Одна строка CSV описывает один материал и его исходную карту Color."""
    if output.exists() or limit < 1:
        raise ValueError("Нужен новый выходной каталог и положительный размер выборки")
    fields = {"material_id", "group_id", "category", "license", "source_url", "color", "caption"}
    rows, ids, images, groups = [], set(), {}, {}
    root = catalog.resolve().parent
    with catalog.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if set(reader.fieldnames or []) != fields:
            raise ValueError(f"Ожидаются колонки: {sorted(fields)}")
        for raw in reader:
            if None in raw or any(not v or not v.strip() for v in raw.values()):
                raise ValueError(f"Неполная запись в строке {reader.line_num}")
            r = {k: v.strip() for k, v in raw.items()}
            if r["license"] != "CC0":
                continue
            mid, group, cat = r["material_id"], r["group_id"], r["category"]
            if not mid.isascii() or not mid.isalnum() or mid.casefold() in ids:
                raise ValueError(f"Недопустимый или повторный material_id: {mid}")
            if r["source_url"] != f"https://ambientcg.com/view?id={mid}":
                raise ValueError(f"Проверьте ссылку на материал {mid}")
            path = (root / r["color"]).resolve()
            if not path.is_relative_to(root) or not path.is_file():
                raise ValueError(f"Не найдена локальная карта Color для {mid}")
            with Image.open(path) as im:
                if im.mode not in ("RGB", "L") or im.width != im.height or im.width < 512:
                    raise ValueError(f"{mid}: нужна квадратная карта RGB или L от 512 пикселей")
                source_mode = im.mode
                low, high = np.percentile(np.asarray(im.convert("L")), [1, 99])
                digest = hashlib.sha256(im.convert("RGB").tobytes()).hexdigest()
            if digest in images:
                raise ValueError(f"Совпадают карты {mid} и {images[digest]}")
            images[digest] = mid
            ids.add(mid.casefold())
            if groups.setdefault(group, cat) != cat:
                raise ValueError(f"Группа {group} отнесена к разным категориям")
            rows.append(dict(r, sha256_rgb=digest, source_mode=source_mode,
                             nearly_uniform_luma=bool(low == high)))
    if len(rows) < limit:
        raise ValueError(f"Доступно {len(rows)} материалов CC0, запрошено {limit}")

    def order(key):
        return hashlib.sha256(f"{seed}/{key}".encode()).digest()

    # Берём по одному материалу каждой категории за проход.
    buckets = {}
    for r in sorted(rows, key=lambda r: order(r["material_id"])):
        buckets.setdefault(r["category"], []).append(r)
    selected = []
    for index in range(max(map(len, buckets.values()))):
        for cat in sorted(buckets):
            if index < len(buckets[cat]):
                selected.append(buckets[cat][index])
        if len(selected) >= limit:
            break
    selected = selected[:limit]
    splits = {}
    for cat in sorted({r["category"] for r in selected}):
        keys = sorted({r["group_id"] for r in selected if r["category"] == cat}, key=order)
        if len(keys) < 3:
            raise ValueError(f"{cat}: нужно хотя бы три независимые группы")
        n = max(1, round(len(keys) / 10))
        for i, key in enumerate(keys):
            splits[key] = "test" if i < n else "validation" if i < 2*n else "train"

    output.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=output.parent) as tmp:
        stage = Path(tmp) / "dataset"
        stage.mkdir()
        with (stage / "manifest.jsonl").open("w", encoding="utf-8") as stream:
            for r in selected:
                dest = Path(splits[r["group_id"]]) / (r["material_id"] + ".png")
                (stage / dest).parent.mkdir(exist_ok=True)
                with Image.open(root / r["color"]) as im:
                    im.convert("RGB").resize((512, 512), Image.Resampling.LANCZOS).save(stage / dest)
                print(json.dumps(dict(r, color=dest.as_posix(), split=dest.parts[0]),
                                 ensure_ascii=False), file=stream)
        stage.rename(output)

In [2]:
from datasets import load_dataset, Image as DatasetImage
from itertools import islice
from tempfile import TemporaryDirectory

repo = "dream-textures/textures-color-1k"
revision = "33c49b76fcf3a18b0d521d8e760c88d49f3e47bc"
data = load_dataset(repo, revision=revision, split="train", streaming=True)
rows = []
with TemporaryDirectory() as tmp:
    root = Path(tmp)
    for record in islice(data.cast_column("image", DatasetImage(decode=False)), 6):
        asset = record["image"]
        name = Path(asset["path"]).name
        mid = Path(name).stem
        (root / name).write_bytes(asset["bytes"])
        rows.append(dict(material_id=mid, group_id=mid, category=mid.rstrip("0123456789"),
                         license="CC0", source_url=f"https://ambientcg.com/view?id={mid}",
                         color=name, caption=record["text"]))
    catalog = root / "catalog.csv"
    with catalog.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    output = root / "prepared"
    prepare(catalog, output, limit=6)
    records = [json.loads(line) for line in (output / "manifest.jsonl").read_text().splitlines()]
    assert len(records) == 6
    for r in records:
        with Image.open(output / r["color"]) as tile:
            assert tile.mode == "RGB" and tile.size == (512, 512)
            assert hashlib.sha256(tile.tobytes()).hexdigest() == r["sha256_rgb"]
    report = [{"id": r["material_id"], "mode": r["source_mode"], "flat": r["nearly_uniform_luma"],
               "split": r["split"]} for r in records]
    print(json.dumps(report, indent=2))
    Path("result.json").write_text(json.dumps(report))

[
  {
    "id": "AcousticFoam003",
    "mode": "L",
    "flat": true,
    "split": "test"
  },
  {
    "id": "Asphalt002",
    "mode": "RGB",
    "flat": false,
    "split": "test"
  },
  {
    "id": "AcousticFoam001",
    "mode": "L",
    "flat": true,
    "split": "validation"
  },
  {
    "id": "Asphalt003",
    "mode": "L",
    "flat": false,
    "split": "validation"
  },
  {
    "id": "AcousticFoam002",
    "mode": "L",
    "flat": true,
    "split": "train"
  },
  {
    "id": "Asphalt001",
    "mode": "RGB",
    "flat": false,
    "split": "train"
  }
]
